# Parser de Extrato Bancário
Lê arquivos no formato  e extrai os lançamentos para DataFrames.

In [1]:
import os
import re
import pandas as pd
from pathlib import Path

In [2]:
def parse_extrato(filepath: str) -> pd.DataFrame:
    with open(filepath, "r", encoding="latin-1") as f:
        lines = f.readlines()

    # Find start and end of transactions
    in_lancamentos = False
    raw_lines = []
    for i, line in enumerate(lines):
        if "Dt. movimento" in line:
            in_lancamentos = True
            continue
        if in_lancamentos:
            # Stop at the separator after transactions end
            if line.strip().startswith("----------"):
                break
            raw_lines.append(line.rstrip("\n"))

    # Parse transactions: main lines start with a date or spaces+date,
    # continuation lines (descrição) start with many spaces and no date pattern
    records = []
    date_pattern = re.compile(r"^\s{2,4}\d{2}/\d{2}/\d{4}")

    for line in raw_lines:
        if not line.strip():
            continue
        if date_pattern.match(line):
            # New transaction line
            records.append({"raw": line, "descricao": ""})
        else:
            # Continuation line (description)
            if records:
                records[-1]["descricao"] = line.strip()

    # Now parse each main line using fixed-width positions
    parsed = []
    for rec in records:
        raw = rec["raw"]
        # Pad line to ensure enough length
        raw = raw.ljust(160)

        dt_movimento = raw[3:13].strip()
        dt_balancete = raw[13:30].strip()
        ag_origem = raw[30:44].strip()
        lote = raw[44:58].strip()
        historico = raw[58:90].strip()
        documento = raw[90:120].strip()
        valor_str = raw[120:133].strip()
        saldo_str = raw[133:].strip()

        # Parse valor
        valor = None
        valor_dc = ""
        m = re.search(r"([\d.,]+)\s*([CD])", valor_str)
        if m:
            valor = float(m.group(1).replace(".", "").replace(",", "."))
            valor_dc = m.group(2)

        # Parse saldo
        saldo = None
        saldo_dc = ""
        m = re.search(r"([\d.,]+)\s*([CD])", saldo_str)
        if m:
            saldo = float(m.group(1).replace(".", "").replace(",", "."))
            saldo_dc = m.group(2)

        parsed.append({
            "dt_movimento": dt_movimento,
            "dt_balancete": dt_balancete,
            "ag_origem": ag_origem,
            "lote": lote,
            "historico": historico,
            "documento": documento,
            "valor": valor,
            "valor_dc": valor_dc,
            "saldo": saldo,
            "saldo_dc": saldo_dc,
            "descricao": rec["descricao"],
        })

    df = pd.DataFrame(parsed)
    return df


def parse_pasta(pasta: str) -> dict[str, pd.DataFrame]:
    """Lê todos os arquivos MMYYYY.txt de uma pasta e retorna dict {MMYYYY: DataFrame}."""
    pattern = re.compile(r"^(\d{6})\.txt$")
    resultado = {}

    for arquivo in sorted(Path(pasta).glob("*.txt")):
        m = pattern.match(arquivo.name)
        if m:
            chave = m.group(1)
            print(f"Processando {arquivo.name}...")
            resultado[chave] = parse_extrato(str(arquivo))

    return resultado


## Processar arquivos da pasta

In [3]:
# Altere para o caminho da sua pasta com os arquivos MMYYYY.txt
cur_dir = Path.cwd()
PASTA = cur_dir / "db/extratos"

dfs = parse_pasta(PASTA)
print(f"Arquivos processados: {len(dfs)}")
for chave in dfs:
    print(f"  {chave}: {len(dfs[chave])} lançamentos")

Processando 012020.txt...
Processando 012021.txt...
Processando 012022.txt...
Processando 012023.txt...
Processando 012024.txt...
Processando 012025.txt...
Processando 012026.txt...
Processando 022020.txt...
Processando 022021.txt...
Processando 022022.txt...
Processando 022023.txt...
Processando 022024.txt...
Processando 022025.txt...
Processando 022026.txt...
Processando 032021.txt...
Processando 032022.txt...
Processando 032023.txt...
Processando 032024.txt...
Processando 032025.txt...
Processando 042021.txt...
Processando 042022.txt...
Processando 042023.txt...
Processando 042024.txt...
Processando 042025.txt...
Processando 052021.txt...
Processando 052022.txt...
Processando 052023.txt...
Processando 052024.txt...
Processando 052025.txt...
Processando 062021.txt...
Processando 062022.txt...
Processando 062023.txt...
Processando 062024.txt...
Processando 062025.txt...
Processando 072021.txt...
Processando 072022.txt...
Processando 072023.txt...
Processando 072024.txt...
Processando 

In [4]:
# Visualizar um extrato específico
chave = list(dfs.keys())[0]  # primeiro arquivo
dfs[chave]

,dt_movimento,dt_balancete,ag_origem,lote,historico,documento,valor,valor_dc,saldo,saldo_dc,descricao
0,30/01/2026,,0000,00000,Saldo Anterior,,NaN,,1171.51,C,
1,02/02/2026,,0000,14134,Recebimentos Diversos,250.000,2144.65,C,3316.16,C,00.394.544/0127-87 MINISTERIO DA SAUDE
2,04/02/2026,,0000,14134,Recebimentos Diversos,24.650,57.68,C,NaN,,TRIBUNAL DE CONTAS DO ESTADO DO RIO GR
3,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.807,630.47,C,NaN,,ESTADO DO RIO GRANDE D
4,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.808,495.43,C,NaN,,ESTADO DO RIO GRANDE D
5,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.811,1295.64,C,NaN,,ESTADO DO RIO GRANDE D
6,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.814,121.81,C,NaN,,ESTADO DO RIO GRANDE D
7,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.815,200.26,C,NaN,,ESTADO DO RIO GRANDE D
8,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.816,124.91,C,NaN,,ESTADO DO RIO GRANDE D
9,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.819,282.27,C,NaN,,ESTADO DO RIO GRANDE D


In [5]:
# Concatenar todos os extratos em um único DataFrame
df_total = pd.concat(
    [df.assign(periodo=k) for k, df in dfs.items()],
    ignore_index=True
)
df_total

,dt_movimento,dt_balancete,ag_origem,lote,historico,documento,valor,valor_dc,saldo,saldo_dc,descricao,periodo
0,30/01/2026,,0000,00000,Saldo Anterior,,NaN,,1171.51,C,,012020
1,02/02/2026,,0000,14134,Recebimentos Diversos,250.000,2144.65,C,3316.16,C,00.394.544/0127-87 MINISTERIO DA SAUDE,012020
2,04/02/2026,,0000,14134,Recebimentos Diversos,24.650,57.68,C,NaN,,TRIBUNAL DE CONTAS DO ESTADO DO RIO GR,012020
3,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.807,630.47,C,NaN,,ESTADO DO RIO GRANDE D,012020
4,04/02/2026,,0000,14138,Ordem Bancária,202.602.030.004.808,495.43,C,NaN,,ESTADO DO RIO GRANDE D,012020
...,...,...,...,...,...,...,...,...,...,...,...,...
1559,24/12/2025,,0000,00000,BB RF Sim SD Diferenciad,942,22471.34,D,0.00,C,BB RF Simples SD Diferenciado,122025
1560,30/12/2025,,0214,99015,Transferência recebida,550.214.000.067.210,148.99,C,NaN,,30/12 11:52 SME DE PARAU FUNDEB,122025
1561,30/12/2025,,0000,14134,Recebimentos Diversos,68.875,57.68,C,NaN,,TRIBUNAL DE CONTAS DO ESTADO DO RIO GR,122025
1562,30/12/2025,,0000,14138,Ordem Bancária,202.512.290.042.411,568.96,C,775.63,C,ESTADO DO RIO GRANDE D,122025


In [6]:
df_total[df_total['valor'].notna() & (round(df_total['valor']) == '1006')]

,dt_movimento,dt_balancete,ag_origem,lote,historico,documento,valor,valor_dc,saldo,saldo_dc,descricao,periodo


In [11]:
df_total.dt_movimento = pd.to_datetime(df_total.dt_movimento, errors='coerce', dayfirst=True)
df_total.dt_balancete = pd.to_datetime(df_total.dt_balancete, errors='coerce', dayfirst=True)

In [12]:
df_total[df_total['valor'] == 1006.59].sort_values(by='dt_movimento').sort_values(by='dt_movimento')

,dt_movimento,dt_balancete,ag_origem,lote,historico,documento,valor,valor_dc,saldo,saldo_dc,descricao,periodo
415,2021-03-26,2021-03-26,0000,14138,Ordem Bancária,202.103.250.030.266,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,032021
528,2021-04-28,2021-04-28,0000,14138,Ordem Bancária,202.104.270.006.533,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,042021
529,2021-04-28,2021-04-28,0000,00000,BB CP Automatico S P,70,1006.59,D,0.00,C,,042021
648,2021-05-27,2021-05-27,0000,14138,Ordem Bancária,202.105.260.001.467,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,052021
760,2021-06-24,2021-06-24,0000,14138,Ordem Bancária,202.106.230.002.670,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,062021
864,2021-07-27,2021-07-27,0000,14138,Ordem Bancária,202.107.260.004.368,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,072021
988,2021-08-26,2021-08-26,0000,14138,Ordem Bancária,202.108.250.002.369,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,082021
1093,2021-09-28,2021-09-28,0000,14138,Ordem Bancária,202.109.270.035.644,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,092021
1200,2021-10-27,2021-10-27,0000,14138,Ordem Bancária,202.110.260.007.082,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,102021
1320,2021-11-26,2021-11-26,0000,14138,Ordem Bancária,202.111.250.001.706,1006.59,C,NaN,,084933710001-64 RIO G DO NORTE ASSEMBL,112021


In [18]:
pgt_alrn_desconto = df_total[df_total['descricao'].str.contains('084933710001') & (df_total['valor'] == 1006.59)].sort_values(by='dt_movimento').sort_values(by='dt_movimento')

In [20]:
len(pgt_alrn_desconto), sum(pgt_alrn_desconto['valor'])

(23, 23151.57)